### **Mounting Drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

### **Inputing the Requirement**

In [ ]:
import datetime as dt, os

# filter config
TIMESTAMP     = dt.datetime.now().strftime('%d%m%Y')
COLUMNS = {
    'Date':      'Date',        # e.g. 'creation_time'
    'Media':     'Media',       # e.g. 'specific_resource_type'
    'Author':    'Author',      # e.g. 'from_username'
    'Mentions':  'Mentions',    # e.g. 'content'
    'Sentiment': 'Sentiment',   # e.g. 'final_sentiment'
    'Link':      'Link',
}
REPORT_TYPE     = 'Monthly' # 'Monthly' or 'Weekly' ('Daily' too for Environment)
ROBJ            = 'Indonesia' # Changes this based on your research object
ISSUES          = 'Environment' # 'Environment' or 'Politics'
PATH            = '/content/drive/MyDrive/Data/Raw Data DPR 2026' # Path to access the data in google drive
FILE_NAME       = 'dpr26_agustus.xlsx'   # one file, or a pattern: '*.xlsx', 'dpr26_*.csv', '*'
KEYWORDS        = []   # [] or None -> no filtering
REQUIRED        = []       # must-have terms (ALL of them); [] -> none
MIN_HITS        = 2                 # how many of KEYWORDS a row must match
EXACT_WORD      = False
DOMINANCE_RATIO = 2.0

### **Notebook Set Up**

In [ ]:
%pip install -q "great[all] @ git+https://github.com/azmkto/GREAT-Tools.git"

import pandas as pd

# Everything that used to be a copy-pasted config cell:
from great import MEDIA_MAP, plat_class, sent_class, plat_colors, sent_colors, classify_issue, resolve_frame, validate_export
from great.viz import checks, prep
from great.viz.overview import monthly_overview, weekly_overview
from great.viz.wordcloud import sentiment_wordclouds
from great.viz.environment import dominant_issue, env_weekly, env_monthly, province_counts, island_counts
from great.viz.style import apply_style

apply_style()

### **Accessing File**

In [ ]:
df = pd.read_excel(f'{PATH}/{FILE_NAME}',header=1)
df1, start_date, end_date = prep.prepare_data(df)

sent_data = prep.sentiment_data(df1)    # Date x Sentiment (wide)
plat_data = prep.platform_data(df1)     # (Date, Media) -> Count
ps_data   = prep.plat_sent_data(df1)    # Media, Sentiment, Count (long)

print(f'{start_date} - {end_date}')

### **Checking Dataframe Info**

In [ ]:
checks.frame_info(df1, 'Main Dataframe')

In [ ]:
checks.frame_info(sent_data, 'Sentiment Dataframe')

In [ ]:
checks.frame_info(plat_data, 'Platform Dataframe')

In [ ]:
checks.frame_info(ps_data, 'Platform-Sentiment Dataframe')

In [ ]:
checks.completeness(df1)

In [ ]:
checks.coverage(df1)

In [ ]:
checks.composition(df1)

In [ ]:
checks.author(df1)

In [ ]:
checks.labels(df1)

### **Visualize**

In [ ]:
if ISSUES == 'Politics':
    if REPORT_TYPE == 'Weekly':
        sent_pct_df, platform_totals_df = weekly_overview(
            ps_data=ps_data,
            platform_data=plat_data,
            sentiment_data=sent_data,
            robj=ROBJ,
            start_date=start_date,
            end_date=end_date,
        )
    else:
        sent_pct_df, platform_totals_df = monthly_overview(
            ps_data=ps_data,
            platform_data=plat_data,
            sentiment_data=sent_data,
            robj=ROBJ,
            start_date=start_date,
            end_date=end_date,
            interval=3,
        )
else:
    df1['Isu_Inti'] = df1['Mentions'].map(classify_issue)
    df1[['Provinsi', 'Pulau']] = resolve_frame(df1)
    if REPORT_TYPE == 'Weekly':
        fig = env_weekly(df1, start_date=start_date, end_date=end_date,
                         dominance_ratio=2.0)   # top issue >= 2x the second -> it becomes a pie
    else:
        fig = env_monthly(df1, start_date=start_date, end_date=end_date, dominance_ratio=2.0)

In [ ]:
if ISSUES == 'Politics'
    if REPORT_TYPE == 'Monthly':
        fig, terms_by_sentiment = sentiment_wordclouds(
            df1,
            sentiments=('positive', 'negative'),
            sent_colors=sent_colors,     # tints each cloud with its sentiment color
            exclude_media=('News',),     # pass () to keep news copy in
            n=200,                       # terms scored per sentiment
            max_words=100,               # terms actually drawn
        )
    else:
        pass
else:
    pass